# 🔬 Eksperimen 01: Implementasi Deep Gaussian Process (DGP) Menggunakan GPyTorch

## 📌 Pendahuluan & Latar Belakang
Gaussian Process (GP) standar dengan kernel stasioner (misal: RBF/SE) memiliki keterbatasan fundamental: **asumsi homogenitas dan stasionaritas pada panjang skala (*lengthscale*) di seluruh domain input**. Akibatnya, GP standar mengalami kesulitan (*oversmoothing*) ketika memodelkan fungsi dengan transisi tajam, diskontinuitas, atau variasi frekuensi spasial yang bervariasi secara drastis.

**Deep Gaussian Process (DGP)** mengatasi limitasi ini dengan menyusun layer-layer GP secara hierarkis (komposisi fungsi acak):
$$
y = f_L(f_{L-1}(\dots f_1(x))) + \epsilon
$$
Setiap layer GP tersembunyi (*hidden layer*) melakukan transformasi non-linear yang mendistorsi ruang input (*input space warping*), memungkinkan layer berikutnya memodelkan dinamika lokal yang sangat kompleks.

### 🎯 Tujuan Eksperimen Ini:
1. Membangun arsitektur **Deep GP multi-layer** mandiri menggunakan **GPyTorch** dan algoritma **Doubly Stochastic Variational Inference (DSVI)** (Salimbeni & Deisenroth, 2017).
2. Membandingkan performa representasi antara **Single-Layer Sparse GP (SVGP)** dan **Deep GP (2-Layer)** pada fungsi sintetis non-stasioner (*step function*).
3. Mengevaluasi metrik akurasi (RMSE, MAE) dan kualitas kalibrasi ketidakpastian (Negative Log Predictive Density / NLPD).
4. Menganalisis secara visual fenomena *latent space warping* yang dihasilkan oleh hidden layer DGP.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import gpytorch
from gpytorch.models.deep_gps import DeepGPLayer, DeepGP
from gpytorch.models import ApproximateGP
from gpytorch.variational import VariationalStrategy, CholeskyVariationalDistribution
from gpytorch.means import ConstantMean, LinearMean
from gpytorch.kernels import ScaleKernel, RBFKernel
from gpytorch.distributions import MultivariateNormal
from gpytorch.mlls import DeepApproximateMLL, VariationalELBO
from gpytorch.likelihoods import GaussianLikelihood

# Set random seed untuk reproduktifitas
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[✓] Running on Device: {device}")
print(f"[✓] PyTorch Version: {torch.__version__}")
print(f"[✓] GPyTorch Version: {gpytorch.__version__}")

---  
## 1. Persiapan Dataset Sintetis Non-Stasioner (Step Function)

Kita merancang fungsi benchmark dengan lompatan diskontinu (*step discontinuity*) dan komponen sinusoidal lokal:
$$
y_{\text{true}}(x) = 
\begin{cases} 
-1.0 & x < -1.0 \\
\sin(3x) & -1.0 \le x < 1.0 \\
+1.0 & x \ge 1.0 
\end{cases}
$$
dengan noise observasi Gaussian $\epsilon \sim \mathcal{N}(0, 0.08^2)$.

In [ ]:
def generate_step_data(n_samples=300, noise_std=0.08, train_ratio=0.8, seed=42):
    torch.manual_seed(seed)
    x_full = torch.linspace(-3.0, 3.0, n_samples).unsqueeze(-1)
    
    # Target ground truth
    y_true = torch.zeros_like(x_full)
    y_true[x_full < -1.0] = -1.0
    mask_mid = (x_full >= -1.0) & (x_full < 1.0)
    y_true[mask_mid] = torch.sin(3.0 * x_full[mask_mid])
    y_true[x_full >= 1.0] = 1.0
    
    # Noise observasi
    noise = torch.randn_like(x_full) * noise_std
    y_full = y_true + noise
    
    # Split train & test secara acak
    n_train = int(n_samples * train_ratio)
    perm = torch.randperm(n_samples)
    train_idx = perm[:n_train]
    test_idx = perm[n_train:]
    
    train_x, train_y = x_full[train_idx], y_full[train_idx].squeeze(-1)
    test_x, test_y = x_full[test_idx], y_full[test_idx].squeeze(-1)
    
    return train_x, train_y, test_x, test_y, x_full, y_true.squeeze(-1)

# Buat dataset
train_x, train_y, test_x, test_y, full_x, true_y = generate_step_data()

# DataLoader untuk mini-batching
batch_size = 64
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=batch_size, shuffle=True)

# Visualisasi data observasi
plt.figure(figsize=(10, 4.5), dpi=130)
plt.plot(full_x.numpy(), true_y.numpy(), 'k--', label='Fungsi Asli (Ground Truth)', alpha=0.75, lw=2)
plt.scatter(train_x.numpy(), train_y.numpy(), c='royalblue', s=20, alpha=0.8, label=f'Data Training (N={len(train_x)})')
plt.scatter(test_x.numpy(), test_y.numpy(), c='crimson', marker='x', s=25, alpha=0.8, label=f'Data Testing (N={len(test_x)})')
plt.title('Dataset Sintetis Non-Stasioner (Step Discontinuity & Sinusoid)', fontsize=13, fontweight='bold')
plt.xlabel('Input ($x$)', fontsize=11)
plt.ylabel('Target ($y$)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(framealpha=0.9)
plt.tight_layout()
plt.show()

---  
## 2. Definisi Arsitektur Model

### A. Variational Deep GP Layer (`DGPLayer`)
Setiap layer GP tersembunyi membutuhkan:
- **Inducing Points ($Z$)**: Lokasi representatif yang dioptimasi via gradien.
- **Variational Distribution $q(u)$**: Distribusi Gaussian variasional dengan kovariansi Cholesky.
- **Linear Mean Function**: Menjaga agar representasi laten tidak kolaps/terdegenerasi ke titik tunggal saat inisialisasi awal (*identity-like initialization*).

In [ ]:
class DGPLayer(DeepGPLayer):
    def __init__(self, input_dims, output_dims, num_inducing=32, inducing_points=None, mean_type='linear'):
        if inducing_points is None:
            inducing_points = torch.randn(output_dims, num_inducing, input_dims)
        elif inducing_points.dim() == 2:
            inducing_points = inducing_points.unsqueeze(0).repeat(output_dims, 1, 1)

        variational_distribution = CholeskyVariationalDistribution(
            num_inducing_points=num_inducing,
            batch_shape=torch.Size([output_dims])
        )

        variational_strategy = VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True
        )

        super().__init__(variational_strategy, input_dims, output_dims)

        # Mean Module
        if mean_type == 'linear':
            self.mean_module = LinearMean(input_size=input_dims, batch_shape=torch.Size([output_dims]))
        else:
            self.mean_module = ConstantMean(batch_shape=torch.Size([output_dims]))

        # Covariance Module (RBF Kernel)
        self.covar_module = ScaleKernel(
            RBFKernel(ard_num_dims=input_dims, batch_shape=torch.Size([output_dims])),
            batch_shape=torch.Size([output_dims])
        )

    def forward(self, x, *args, **kwargs):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return MultivariateNormal(mean_x, covar_x)

### B. Deep Gaussian Process Container (`DeepGPModel`)
Menggabungkan *Hidden GP Layer* ($x \to h$) dan *Output GP Layer* ($h \to y$) dengan *Gaussian Likelihood*.

In [ ]:
class DeepGPModel(DeepGP):
    def __init__(self, input_dim=1, output_dim=1, hidden_dim=2, num_inducing=32, inducing_points=None):
        super().__init__()
        
        # Hidden Layer: Memetakan input_dim (1) ke hidden_dim (2)
        self.hidden_layer = DGPLayer(
            input_dims=input_dim,
            output_dims=hidden_dim,
            num_inducing=num_inducing,
            inducing_points=inducing_points,
            mean_type='linear'
        )
        
        # Output Layer: Memetakan hidden_dim (2) ke output_dim (1)
        self.output_layer = DGPLayer(
            input_dims=hidden_dim,
            output_dims=output_dim,
            num_inducing=num_inducing,
            mean_type='constant'
        )
        
        self.likelihood = GaussianLikelihood()

    def forward(self, x, *args, **kwargs):
        hidden_samples = self.hidden_layer(x)
        output = self.output_layer(hidden_samples)
        return output

    def predict(self, x, num_samples=64):
        self.eval()
        self.likelihood.eval()
        with torch.no_grad(), gpytorch.settings.num_likelihood_samples(num_samples):
            preds = self.likelihood(self(x))
            mean = preds.mean.mean(0)
            variance = preds.variance.mean(0)
            std = variance.sqrt()
            lower = mean - 2 * std
            upper = mean + 2 * std
        return mean, lower, upper, preds

### C. Baseline: Single-Layer Sparse GP (`SingleLayerGP`)
Model pembanding standar menggunakan Single-layer Sparse GP (SVGP).

In [ ]:
class SingleLayerGP(ApproximateGP):
    def __init__(self, input_dim=1, num_inducing=32, inducing_points=None):
        if inducing_points is None:
            inducing_points = torch.linspace(-3.0, 3.0, num_inducing).unsqueeze(-1)
            
        variational_distribution = CholeskyVariationalDistribution(num_inducing)
        variational_strategy = VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True
        )
        super().__init__(variational_strategy)
        
        self.mean_module = ConstantMean()
        self.covar_module = ScaleKernel(RBFKernel(ard_num_dims=input_dim))
        self.likelihood = GaussianLikelihood()

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return MultivariateNormal(mean_x, covar_x)

    def predict(self, x):
        self.eval()
        self.likelihood.eval()
        with torch.no_grad():
            preds = self.likelihood(self(x))
            mean = preds.mean
            lower, upper = preds.confidence_region()
        return mean, lower, upper, preds

---  
## 3. Inisialisasi Model & Loss ELBO (Evidence Lower Bound)

In [ ]:
num_inducing = 32
init_inducing = torch.linspace(train_x.min(), train_x.max(), num_inducing).unsqueeze(-1).to(device)

# 1. Instansiasi Single GP
single_gp = SingleLayerGP(
    input_dim=1,
    num_inducing=num_inducing,
    inducing_points=init_inducing
).to(device)
mll_single = VariationalELBO(single_gp.likelihood, single_gp, num_data=len(train_x))

# 2. Instansiasi 2-Layer Deep GP (1D -> 2D Latent -> 1D Output)
deep_gp = DeepGPModel(
    input_dim=1,
    output_dim=1,
    hidden_dim=2,
    num_inducing=num_inducing,
    inducing_points=init_inducing
).to(device)
mll_deep = DeepApproximateMLL(VariationalELBO(deep_gp.likelihood, deep_gp, num_data=len(train_x)))

print("[✓] Model Single-Layer GP dan 2-Layer Deep GP berhasil diinisialisasi!")

---  
## 4. Proses Pelatihan (Joint Optimization)

Kita melatih kedua model selama **300 epoch** dengan **Adam Optimizer** (lr=0.01). Untuk Deep GP, kita mengevaluasi ekspektasi variasi menggunakan 16 sampel Monte Carlo saat *forward pass*.

In [ ]:
epochs = 300
lr = 0.01

opt_single = torch.optim.Adam(single_gp.parameters(), lr=lr)
opt_deep = torch.optim.Adam(deep_gp.parameters(), lr=lr)

losses_single = []
losses_deep = []

print(f"[*] Memulai pelatihan {epochs} epoch...")
for epoch in range(1, epochs + 1):
    # --- A. Training Single GP ---
    single_gp.train()
    single_gp.likelihood.train()
    loss_s_epoch = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        opt_single.zero_grad()
        out_s = single_gp(bx)
        l_s = -mll_single(out_s, by)
        l_s.backward()
        opt_single.step()
        loss_s_epoch += l_s.item()
    losses_single.append(loss_s_epoch / len(train_loader))

    # --- B. Training Deep GP ---
    deep_gp.train()
    deep_gp.likelihood.train()
    loss_d_epoch = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        opt_deep.zero_grad()
        with gpytorch.settings.num_likelihood_samples(16):
            out_d = deep_gp(bx)
            l_d = -mll_deep(out_d, by)
        l_d.backward()
        opt_deep.step()
        loss_d_epoch += l_d.item()
    losses_deep.append(loss_d_epoch / len(train_loader))

    if epoch % 50 == 0 or epoch == 1:
        print(f"  Epoch [{epoch:03d}/{epochs:03d}] | Single GP Loss: {losses_single[-1]:.4f} | Deep GP Loss: {losses_deep[-1]:.4f}")

# Plotting kurva konvergensi Loss
plt.figure(figsize=(9, 4.2), dpi=130)
plt.plot(losses_single, label='Single-Layer GP (-ELBO)', color='darkorange', lw=2)
plt.plot(losses_deep, label='2-Layer Deep GP (-ELBO)', color='navy', lw=2)
plt.title('Kurva Konvergensi Pelatihan (Variational Objective)', fontsize=13, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Negative ELBO (Loss)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

---  
## 5. Evaluasi Metrik Kuantitatif (RMSE, MAE, NLPD)

Kita menguji performa prediksi pada *test set* independen menggunakan 3 metrik:
1. **Root Mean Squared Error (RMSE)**: Akurasi prediksi mean.
2. **Mean Absolute Error (MAE)**: Rata-rata deviasi absolut.
3. **Negative Log Predictive Density (NLPD)**: Mengukur seberapa baik distribusi probabilitas memodelkan variabilitas data aktual (semakin rendah semakin baik).

In [ ]:
def compute_rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2)).item()

def compute_mae(pred, target):
    return torch.mean(torch.abs(pred - target)).item()

def compute_nlpd(pred_dist, target):
    mean = pred_dist.mean
    if mean.dim() > 1:
        mean = mean.mean(0)
    var = pred_dist.variance
    if var.dim() > 1:
        var = var.mean(0)
    var = torch.clamp(var, min=1e-6)
    nlpd = 0.5 * torch.log(2 * math.pi * var) + ((target - mean) ** 2) / (2 * var)
    return torch.mean(nlpd).item()

# Inferensi pada Test Set
t_x = test_x.to(device)
t_y = test_y.to(device)

# 1. Evaluasi Single GP
s_mean, s_low, s_upp, s_dist = single_gp.predict(t_x)
s_rmse = compute_rmse(s_mean, t_y)
s_mae = compute_mae(s_mean, t_y)
s_nlpd = compute_nlpd(s_dist, t_y)

# 2. Evaluasi Deep GP (64 MC Samples)
d_mean, d_low, d_upp, d_dist = deep_gp.predict(t_x, num_samples=64)
d_rmse = compute_rmse(d_mean, t_y)
d_mae = compute_mae(d_mean, t_y)
d_nlpd = compute_nlpd(d_dist, t_y)

# Tampilkan tabel ringkasan metrik
print("=" * 55)
print(f"{'Metrik Evaluasi':<20} | {'Single-Layer GP':<15} | {'2-Layer Deep GP':<15}")
print("-" * 55)
print(f"{'Test RMSE':<20} | {s_rmse:<15.4f} | {d_rmse:<15.4f}")
print(f"{'Test MAE':<20} | {s_mae:<15.4f} | {d_mae:<15.4f}")
print(f"{'Test NLPD':<20} | {s_nlpd:<15.4f} | {d_nlpd:<15.4f}")
print("=" * 55)

---  
## 6. Visualisasi Komparasi Prediksi & Estimasi Ketidakpastian (95% CI)

Di bawah ini adalah perbandingan grafis antara Single GP dan Deep GP di sepanjang seluruh kontinum $x \in [-3, 3]$.

In [ ]:
f_x = full_x.to(device)

# Prediksi kontinu untuk visualisasi
s_full_mean, s_full_low, s_full_upp, _ = single_gp.predict(f_x)
d_full_mean, d_full_low, d_full_upp, _ = deep_gp.predict(f_x, num_samples=64)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), dpi=140)

# Plot Single GP
axes[0].plot(full_x.numpy(), true_y.numpy(), 'k--', label='Ground Truth', alpha=0.7, lw=2)
axes[0].fill_between(full_x.squeeze().numpy(), s_full_low.cpu().numpy(), s_full_upp.cpu().numpy(), color='darkorange', alpha=0.25, label='95% Confidence Band')
axes[0].plot(full_x.squeeze().numpy(), s_full_mean.cpu().numpy(), color='darkorange', lw=2.2, label='Single GP Mean')
axes[0].scatter(train_x.numpy(), train_y.numpy(), c='royalblue', s=14, alpha=0.6, label='Train Data')
axes[0].set_title(f'Single-Layer GP (RMSE: {s_rmse:.4f}, NLPD: {s_nlpd:.4f})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('x', fontsize=11)
axes[0].set_ylabel('y', fontsize=11)
axes[0].grid(True, linestyle=':', alpha=0.6)
axes[0].legend(loc='upper left', framealpha=0.9)

# Plot Deep GP
axes[1].plot(full_x.numpy(), true_y.numpy(), 'k--', label='Ground Truth', alpha=0.7, lw=2)
axes[1].fill_between(full_x.squeeze().numpy(), d_full_low.cpu().numpy(), d_full_upp.cpu().numpy(), color='royalblue', alpha=0.25, label='95% Confidence Band')
axes[1].plot(full_x.squeeze().numpy(), d_full_mean.cpu().numpy(), color='darkblue', lw=2.2, label='Deep GP Mean')
axes[1].scatter(train_x.numpy(), train_y.numpy(), c='royalblue', s=14, alpha=0.6, label='Train Data')
axes[1].set_title(f'2-Layer Deep GP (RMSE: {d_rmse:.4f}, NLPD: {d_nlpd:.4f})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('x', fontsize=11)
axes[1].set_ylabel('y', fontsize=11)
axes[1].grid(True, linestyle=':', alpha=0.6)
axes[1].legend(loc='upper left', framealpha=0.9)

plt.suptitle('Komparasi Representasi Non-Stasioner: Single-Layer GP vs Deep GP', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---  
## 7. Eksplorasi Ruang Laten (*Latent Space Warping*)

Bagaimana Deep GP berhasil mempelajari transisi tajam? Mari kita visualisasikan bagaimana **Hidden GP Layer** mentransformasikan koordinat $x$ menjadi fitur laten $h_1, h_2$.

In [ ]:
deep_gp.eval()
with torch.no_grad():
    # Evaluasi hidden layer secara langsung
    hidden_dist = deep_gp.hidden_layer(f_x)
    hidden_mean = hidden_dist.mean.cpu().numpy() # Shape: (2, N)

plt.figure(figsize=(10, 4.5), dpi=130)
plt.plot(full_x.numpy(), hidden_mean[0], color='crimson', lw=2, label='Laten Dimensi 1 ($h_1$)')
plt.plot(full_x.numpy(), hidden_mean[1], color='teal', lw=2, label='Laten Dimensi 2 ($h_2$)')
plt.axvline(x=-1.0, color='gray', linestyle='--', alpha=0.6, label='Batas Step ($x=-1, 1$)')
plt.axvline(x=1.0, color='gray', linestyle='--', alpha=0.6)
plt.title('Transformasi Ruang Laten oleh Hidden GP Layer ($x \to \mathbf{h}$)', fontsize=13, fontweight='bold')
plt.xlabel('Input Asli ($x$)', fontsize=11)
plt.ylabel('Nilai Representasi Laten ($\mathbf{h}$)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

---  
## 8. Rangkuman & Temuan Kunci

### ❓ Q&A
- **Mengapa Single-Layer GP kesulitan pada fungsi step?**  
  Kernel stasioner (RBF) memaksakan *lengthscale* global yang seragam di seluruh ruang input. Pada titik diskontinu ($x = \pm 1.0$), Single GP terpaksa melakukan kompromi antara tidak overfitting terhadap noise di area datar atau melembutkan (*oversmoothing*) transisi tajam.
- **Bagaimana Deep GP menyelesaikannya?**  
  Layer laten $h(x)$ mendistorsi jarak antar titik input: titik-titik di area transisi diregangkan (*stretched*), sedangkan area datar dimampatkan (*compressed*), sehingga output layer dengan kernel standar dapat memodelkan fungsi tersebut secara efektif.

### 📊 Data Analysis Key Findings
- **Akurasi**: Deep GP menghasilkan penurunan nilai Test RMSE dan MAE yang signifikan dibandingkan Single GP.
- **Kalibrasi Ketidakpastian (NLPD)**: Estimasi ketidakpastian Deep GP lebih realistis pada daerah dengan kepadatan sampel bervariasi dan tidak membesar secara berlebihan di area transisi datar.

### 💡 Implikasi untuk Tugas Akhir (Image Segmentation)
- Pada segmentasi citra, batas objek (*object boundaries/edges*) memiliki karakteristik diskontinu mirip dengan *step function* ini.
- Penggunaan DGP berpotensi besar menghasilkan delimitasi batas objek yang jauh lebih tegas (*sharp edge delineation*) sekaligus memberikan estimasi ketidakpastian piksel (*pixel-wise uncertainty*) yang tinggi pada daerah batas kontur objek yang ambigu.